<a href="https://colab.research.google.com/github/vencov/FAV_course/blob/main/TutRatioPitch/Tut_Ratio_PitchExp_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Measuring Your Own Mel Scale: A Forced-Choice Pitch Experiment


# BE CAREFUL IF YOU DECIDE TO USE HEADPHONES, SET THE VOLUME TO MINIMUM AT THE BEGINING AND THEN TRY TO ADJUST TO GET COMFORTABLE LEVEL. THE RESULT SHOULD NOT DEPEND STRONGLY ON INTENSITY, SO PERFORM THE EXPERIMENT ONLY USING COMFORTABLE LOUDNESS LEVEL AND KEEP IN MIND THAT THE LOUDNESS WILL CHANGE WITH FREQUENCY!!!

In this version of the experiment, you will **not** see or set any frequency numbers —
that avoids the temptation to just "halve the Hz value" instead of judging perceived pitch.

Instead, for each trial you will:
1. Hear a **reference tone**.
2. Hear a **test tone**.
3. Judge whether the test tone sounds like **more than half as high**, or
   **less than half as high**, as the reference — using two buttons.

You will do this many times, with test tones spread across a range of frequencies for
each reference. Afterwards, we'll look at what *fraction* of the time you judged each
test tone as "higher than half" — this traces out a **psychometric curve**, and the point
where you're split 50/50 is your personal "half as high" match, found *without* you ever
adjusting a slider toward a suspiciously round number.

Use headphones if you can.


In [7]:
import numpy as np
import random
from ipywidgets import Button, Output, VBox, HBox, Label, Layout
from IPython.display import Audio, display, clear_output

SAMPLE_RATE = 44100

def pure_tone(freq_hz, duration_s=0.8, amp=0.1, sr=SAMPLE_RATE):
    t = np.linspace(0, duration_s, int(sr * duration_s), endpoint=False)
    fade = int(0.01 * sr)
    envelope = np.ones_like(t)
    envelope[:fade] = np.linspace(0, 1, fade)
    envelope[-fade:] = np.linspace(1, 0, fade)
    return amp * envelope * np.sin(2 * np.pi * freq_hz * t)

def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

def mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)


## 1. Building the trial list

For each reference frequency, we generate several **test tones** spread evenly in *mel*
around the predicted "half as high" point (not evenly in Hz — that would bias the range
towards the low end).

To keep the task manageable, trials are grouped into **blocks**: all test tones for one
reference are presented together, one after another, before moving on to the next
reference. The **order of the blocks** is randomized (so it's not always the same
reference that goes first), and the test tones **within** each block are also shuffled —
but you won't be told the reference's frequency in Hz, so there's still nothing to anchor
a numeric guess on.


In [3]:
REFERENCE_FREQS = [600, 800, 1600, 3200, 6400]
N_TEST_TONES = 7      # test tones per reference (= trials per block)
SPREAD_FRAC = 0.35    # how far the test tones spread around the predicted half-mel point

def generate_test_freqs(reference_hz, n=N_TEST_TONES, spread_frac=SPREAD_FRAC):
    mel_ref = hz_to_mel(reference_hz)
    mel_half = mel_ref / 2
    delta = spread_frac * mel_ref
    mel_values = np.linspace(mel_half - delta, mel_half + delta, n)
    mel_values = np.clip(mel_values, hz_to_mel(50), mel_ref - 50)
    return mel_to_hz(mel_values)

blocks = []
for ref in REFERENCE_FREQS:
    block_trials = [{'reference_hz': ref, 'test_hz': float(tf)} for tf in generate_test_freqs(ref)]
    random.shuffle(block_trials)   # shuffle order of test tones within this block
    blocks.append(block_trials)

random.shuffle(blocks)             # shuffle which reference's block comes first
trials = [t for block in blocks for t in block]

print(f"Total trials: {len(trials)}  ({len(blocks)} blocks of {N_TEST_TONES} trials each)")


Total trials: 35  (5 blocks of 7 trials each)


BE CAREFUL IF YOU DECIDE TO USE HEADPHONES, SET THE VOLUME TO MINIMUM AT THE BEGINING AND THEN TRY TO ADJUST TO GET COMFORTABLE LOUDNESS. THE RESULT SHOULD NOT DEPEND STRONGLY ON INTENSITY, SO PERFORM THE EXPERIMENT ONLY USING COMFORTABLE LOUDNESS LEVEL AND KEEP IN MIND THAT THE LOUDNESS WILL CHANGE WITH FREQUENCY!!!

## 2. Run the experiment

For each trial: click **Play reference**, then **Play test tone**, then judge using the
two buttons. The next trial loads automatically after you respond — there's no way to
change your answer afterwards, so trust your first impression.

Since the reference tone stays the same for all 7 trials in a block, you only need to
"remember" one reference pitch at a time before it switches to the next block.


In [ ]:
results = []
trial_index = {'i': 0}

progress_label = Label()
play_ref_btn = Button(description='Play reference')
play_test_btn = Button(description='Play test tone')
higher_btn = Button(description='Test sounds MORE than half as high', button_style='info',
                     layout=Layout(width='260px'))
lower_btn = Button(description='Test sounds LESS than half as high', button_style='warning',
                    layout=Layout(width='260px'))
out = Output()

def update_progress():
    i = trial_index['i']
    if i < len(trials):
        block_num = i // N_TEST_TONES + 1
        trial_in_block = i % N_TEST_TONES + 1
        progress_label.value = (f"Block {block_num}/{len(blocks)}, trial {trial_in_block}/{N_TEST_TONES} "
                                 f"(same reference tone throughout this block) — "
                                 f"play both tones, then judge.")
    else:
        progress_label.value = "All trials complete! Run the next cells to see your results."

def play_ref(_):
    with out:
        clear_output(wait=True)
        i = trial_index['i']
        if i < len(trials):
            display(Audio(pure_tone(trials[i]['reference_hz']), rate=SAMPLE_RATE,
                           autoplay=True, normalize=False))

def play_test(_):
    with out:
        clear_output(wait=True)
        i = trial_index['i']
        if i < len(trials):
            display(Audio(pure_tone(trials[i]['test_hz']), rate=SAMPLE_RATE,
                           autoplay=True, normalize=False))

def respond(response):
    i = trial_index['i']
    if i < len(trials):
        results.append({
            'reference_hz': trials[i]['reference_hz'],
            'test_hz': trials[i]['test_hz'],
            'response': response,
        })
        trial_index['i'] += 1
        update_progress()
        with out:
            clear_output(wait=True)

play_ref_btn.on_click(play_ref)
play_test_btn.on_click(play_test)
higher_btn.on_click(lambda _: respond('higher'))
lower_btn.on_click(lambda _: respond('lower'))

update_progress()
display(VBox([
    progress_label,
    HBox([play_ref_btn, play_test_btn]),
    HBox([higher_btn, lower_btn]),
    out,
]))


## 3. Look at your psychometric curves

For each reference frequency, this plots the **fraction of "higher" responses** against
the test tone frequency. Where your curve crosses **50%** is your personal "half as high"
match — found from your pattern of judgments, not from a slider position.


In [ ]:
import matplotlib.pyplot as plt

assert len(results) == len(trials), "Complete all trials above before continuing."

fig, axes = plt.subplots(1, len(REFERENCE_FREQS), figsize=(4 * len(REFERENCE_FREQS), 4), sharey=True)

your_matches = {}

for ax, ref in zip(axes, REFERENCE_FREQS):
    ref_results = [r for r in results if r['reference_hz'] == ref]
    test_freqs_sorted = sorted(set(r['test_hz'] for r in ref_results))
    proportions = []
    for tf in test_freqs_sorted:
        responses = [r['response'] for r in ref_results if r['test_hz'] == tf]
        proportions.append(responses.count('higher') / len(responses))

    ax.plot(test_freqs_sorted, proportions, 'o-', color='C0')
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)

    # Estimate the 50% crossover point (PSE) by linear interpolation
    pse = None
    for j in range(len(proportions) - 1):
        p1, p2 = proportions[j], proportions[j + 1]
        if (p1 - 0.5) * (p2 - 0.5) <= 0 and p1 != p2:
            f1, f2 = test_freqs_sorted[j], test_freqs_sorted[j + 1]
            pse = f1 + (0.5 - p1) * (f2 - f1) / (p2 - p1)
            break
    your_matches[ref] = pse
    if pse is not None:
        ax.axvline(pse, color='C1', linestyle=':', label=f'Your match: {pse:.0f} Hz')
        ax.legend(fontsize=8)

    predicted_hz = mel_to_hz(hz_to_mel(ref) / 2)
    ax.axvline(predicted_hz, color='green', linestyle='-.', alpha=0.6)

    ax.set_title(f'Ref: {ref} Hz')
    ax.set_xlabel('Test tone (Hz)')

axes[0].set_ylabel('Proportion "higher than half" responses')
plt.tight_layout()
plt.show()

print("Your estimated 'half as high' matches vs. the standard mel scale prediction:")
for ref in REFERENCE_FREQS:
    predicted_hz = mel_to_hz(hz_to_mel(ref) / 2)
    match = your_matches[ref]
    match_str = f"{match:.0f} Hz" if match is not None else "not determined (curve never crossed 50%)"
    print(f"Reference {ref:6.0f} Hz  ->  your match: {match_str:>20}   (mel scale predicts {predicted_hz:.0f} Hz)")


## 4. Save your results

Run the cell below to save your raw trial-by-trial responses to a `.pkl` file. The file
will download automatically — submit that downloaded file as your assignment.


In [6]:
import pickle
from datetime import datetime

student_id = input("Enter your username: ").strip()

data_to_save = {
    'student_id': student_id,
    'timestamp': datetime.now().isoformat(),
    'reference_freqs': REFERENCE_FREQS,
    'trials': trials,
    'results': results,
}

filename = f"mel_2afc_{student_id.replace(' ', '_')}.pkl"
with open(filename, 'wb') as f:
    pickle.dump(data_to_save, f)

print(f"Saved to {filename}")

try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print("Not running in Colab — file saved locally, no auto-download triggered.")


Enter your username: vv
Saved to mel_2afc_vv.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Reflection questions

1. Why might asking "is this tone more or less than half as high?" (a forced choice)
   produce more honest data than asking you to *set* a tone to exactly half as high with
   a slider?
2. Look at your psychometric curves — are some steeper than others? A steep curve near
   50% means your judgments were consistent; a shallow, noisy curve means you were
   uncertain across a wide range of test tones. Which reference frequencies gave you the
   most consistent judgments?
3. Compare your estimated matches (orange dotted line) to the standard mel-scale
   prediction (green dot-dashed line) in each panel. Do they agree better at some
   reference frequencies than others?
4. Notice that with only one response per test tone, your psychometric curve is based on very little
   data per point. Think how to change the experiment design (e.g. number of
   repetitions, number of test tones, spacing of test tones) to get a more reliable
   estimate of your personal "half as high" point?
